# Layer 2: Post-Quantum Cryptography (PQC) Layer

This notebook demonstrates the PQC layer for verifying bank messages using NIST FIPS 204 (ML-DSA) digital signatures.

### Local Prerequisites (Windows):
1. **Install liboqs**: Download the binary or build from [OpenQuantumSafe/liboqs](https://github.com/open-quantum-safe/liboqs).
2. **Install liboqs-python**: Execute `pip install liboqs-python` in your terminal.
3. **Path Setup**: Ensure `liboqs.dll` is in your system PATH or the same directory as your Python executable.

In [ ]:
import os
import sys
import json
import time
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np

# Set base path to current directory for local execution
base_path = os.getcwd()

# Initialize local project structure
folders = [
    os.path.join(base_path, 'keys'),
    os.path.join(base_path, 'test_cases'),
    os.path.join(base_path, 'results')
]

for folder in folders:
    os.makedirs(folder, exist_ok=True)
    print(f"Directory ready: {folder}")

try:
    import oqs
    print("\nSuccess: oqs module imported correctly.")
    print("Enabled Signatures:", oqs.get_enabled_sig_mechanisms()[:5], "...")
except ImportError:
    print("\nERROR: liboqs-python not found. Run 'pip install liboqs-python'.")
except Exception as e:
    print(f"\nERROR: Could not initialize liboqs: {e}")

## 1. Key Generation
In a production system, the bank (SBI) generates a PQC keypair. The Private Key stays in a secure HSM, while the Public Key is distributed to the mobile app for verification.

In [ ]:
import oqs

keys_path = os.path.join(base_path, 'keys')
sig_alg = 'ML-DSA-65'

with oqs.Signature(sig_alg) as signer:
    print(f"Generating {sig_alg} keypair...")
    public_key = signer.generate_keypair()
    private_key = signer.export_secret_key()

    # Save Public Key
    pub_file = os.path.join(keys_path, f'bank_{sig_alg}_pub.key')
    with open(pub_file, 'wb') as f:
        f.write(public_key)

    # Save Private Key
    priv_file = os.path.join(keys_path, f'bank_{sig_alg}_priv.key')
    with open(priv_file, 'wb') as f:
        f.write(private_key)

    print(f"Keys successfully saved to: {keys_path}")

## 2. Bank SMS Gateway Simulation
This class simulates the bank's backend service that signs SMS messages before dispatching them.

In [ ]:
class BankSMSGateway:
    def __init__(self, algorithm='ML-DSA-65'):
        self.algorithm = algorithm
        self.keys_path = os.path.join(base_path, 'keys')
        self.pub_file = os.path.join(self.keys_path, f'bank_{algorithm}_pub.key')
        self.priv_file = os.path.join(self.keys_path, f'bank_{algorithm}_priv.key')

        if not os.path.exists(self.pub_file):
            raise FileNotFoundError(f"Keys not found at {self.pub_file}. Run the Key Generation cell first.")
        
        with open(self.pub_file, 'rb') as f: self.public_key = f.read()
        with open(self.priv_file, 'rb') as f: self.private_key = f.read()
        print(f"Bank Gateway initialized with {self.algorithm}")

    def sign_sms(self, sender, message):
        payload = {
            "sender": sender,
            "message": message,
            "timestamp": int(time.time())
        }
        payload_bytes = json.dumps(payload, sort_keys=True).encode('utf-8')
        
        with oqs.Signature(self.algorithm) as signer:
            signer.secret_key = self.private_key
            signature = signer.sign(payload_bytes)

        return {
            "payload": payload,
            "signature": signature.hex(),
            "algorithm": self.algorithm
        }

## 3. Generate Evaluation Test Cases
We create a suite of 8 scenarios including legitimate messages, tampering, replay attacks, and spoofing.

In [ ]:
print("Generating 8 attack and legitimate scenarios...")
gateway = BankSMSGateway()
tc_path = os.path.join(base_path, 'test_cases')

# TC1: Legitimate
tc1 = gateway.sign_sms("SBI-ALERTS", "Your OTP for SBI login is 482910.")
with open(os.path.join(tc_path, 'tc1_legitimate.json'), 'w') as f: json.dump(tc1, f, indent=2)

# TC2: Fraud (No Signature)
tc2 = {"payload": {"sender": "SBI-ALERTS", "message": "URGENT: Verify at bit.ly/sbi", "timestamp": int(time.time())}, "signature": None}
with open(os.path.join(tc_path, 'tc2_fraud_unsigned.json'), 'w') as f: json.dump(tc2, f, indent=2)

# TC3: Tampered (Message changed after signing)
tc3 = gateway.sign_sms("SBI-ALERTS", "OTP 482910")
tc3['payload']['message'] = "OTP 999999"
with open(os.path.join(tc_path, 'tc3_tampered.json'), 'w') as f: json.dump(tc3, f, indent=2)

# TC4: Clever Fake (No signature, text looks real)
tc4 = {"payload": {"sender": "SBI-ALERTS", "message": "Security update successful.", "timestamp": int(time.time())}, "signature": None}
with open(os.path.join(tc_path, 'tc4_clever_fake.json'), 'w') as f: json.dump(tc4, f, indent=2)

# TC5: Jamming (Legitimate sign, but flagged externally)
tc5 = gateway.sign_sms("SBI-ALERTS", "OTP 739201")
tc5['jamming_scenario'] = True
with open(os.path.join(tc_path, 'tc5_jamming.json'), 'w') as f: json.dump(tc5, f, indent=2)

# TC6: Wrong Key (Spoofed signature with different private key)
with oqs.Signature('ML-DSA-65') as fake_signer:
    fake_signer.generate_keypair()
    payload6 = {"sender": "SBI-ALERTS", "message": "Spoofed!", "timestamp": int(time.time())}
    sig6 = fake_signer.sign(json.dumps(payload6, sort_keys=True).encode())
    tc6 = {"payload": payload6, "signature": sig6.hex(), "algorithm": "ML-DSA-65"}
with open(os.path.join(tc_path, 'tc6_wrong_key.json'), 'w') as f: json.dump(tc6, f, indent=2)

# TC7: Replay Attack (Expired timestamp)
old_payload = {"sender": "SBI-ALERTS", "message": "OTP 112233", "timestamp": int(time.time()) - 7200}
with oqs.Signature('ML-DSA-65') as signer:
    signer.secret_key = gateway.private_key
    sig7 = signer.sign(json.dumps(old_payload, sort_keys=True).encode())
    tc7 = {"payload": old_payload, "signature": sig7.hex(), "algorithm": "ML-DSA-65"}
with open(os.path.join(tc_path, 'tc7_expired_timestamp.json'), 'w') as f: json.dump(tc7, f, indent=2)

# TC8: Garbage Signature
tc8 = tc1.copy()
tc8['signature'] = "deadbeef12345garbage"
with open(os.path.join(tc_path, 'tc8_garbage_signature.json'), 'w') as f: json.dump(tc8, f, indent=2)

print("All 8 test cases created successfully.")

## 4. PQC Verification Logic
This is the core function that will run on the receiver side to validate the message.

In [ ]:
def get_pqc_score(packet, public_key):
    if not packet.get('signature'):
        return {'pqc_score': 0.0, 'valid': False, 'status': 'No signature found'}

    try:
        alg = packet.get('algorithm', 'ML-DSA-65')
        with oqs.Signature(alg) as verifier:
            payload_bytes = json.dumps(packet['payload'], sort_keys=True).encode('utf-8')
            sig_bytes = bytes.fromhex(packet['signature'])
            is_valid = verifier.verify(payload_bytes, sig_bytes, public_key)

        if is_valid:
            # Check timestamp (expiry: 10 mins)
            if (int(time.time()) - packet['payload'].get('timestamp', 0)) > 600:
                return {'pqc_score': 0.0, 'valid': False, 'status': 'Expired (Replay)'}
            return {'pqc_score': 1.0, 'valid': True, 'status': 'Authentic signature'}
        else:
            return {'pqc_score': 0.0, 'valid': False, 'status': 'Invalid signature'}
    except Exception as e:
        return {'pqc_score': 0.0, 'valid': False, 'status': f'Error: {str(e)}'}

## 5. Run Scenarios and Results

In [ ]:
test_files = [
    ('tc1_legitimate.json', 'LEGITIMATE'), ('tc2_fraud_unsigned.json', 'FRAUD'),
    ('tc3_tampered.json', 'FRAUD'), ('tc4_clever_fake.json', 'FRAUD'),
    ('tc5_jamming.json', 'LEGITIMATE'), ('tc6_wrong_key.json', 'FRAUD'),
    ('tc7_expired_timestamp.json', 'FRAUD'), ('tc8_garbage_signature.json', 'FRAUD')
]

results = []
pub_key = gateway.public_key

for filename, expected in test_files:
    with open(os.path.join(tc_path, filename)) as f:
        packet = json.load(f)
    res = get_pqc_score(packet, pub_key)
    verdict = 'LEGITIMATE' if res['valid'] else 'FRAUD'
    results.append({
        'Test Case': filename.split('.')[0], 'Verdict': verdict, 
        'Expected': expected, 'Status': res['status'], 'Pass': (verdict == expected)
    })

df_res = pd.DataFrame(results)
print(df_res.to_string(index=False))
df_res.to_csv(os.path.join(base_path, 'results', 'scenario_results.csv'), index=False)

## 6. Benchmarking
We compare ML-DSA levels against classical algorithms.

In [ ]:
bench_data = []
msg = b"Your OTP is 123456"

for algo in ['ML-DSA-44', 'ML-DSA-65', 'ML-DSA-87']:
    with oqs.Signature(algo) as s:
        pk = s.generate_keypair()
        start = time.time()
        for _ in range(100): sig = s.sign(msg)
        sign_time = (time.time() - start) / 100 * 1000
        
        start = time.time()
        for _ in range(100): s.verify(msg, sig, pk)
        verify_time = (time.time() - start) / 100 * 1000
        
        bench_data.append({
            'Algorithm': algo, 'Sign (ms)': sign_time, 'Verify (ms)': verify_time,
            'Quantum Safe': 'Yes'
        })

df_bench = pd.DataFrame(bench_data)
df_bench.to_csv(os.path.join(base_path, 'results', 'benchmark.csv'), index=False)
print(df_bench)